# ScreamingFace · discover models and benchmarks

Discover what this ScreamingFace engine can run without mixing discovery with execution:

- executable model IDs come from its model routes;
- canonical benchmark IDs come from its benchmark manifests.

This notebook shows both lists and their shared filters, then makes the separate benchmark-loading
boundary explicit.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

Docker is the only prerequisite. Listing or loading a manifest does not call a model or download
benchmark cases, so no provider or dataset credentials are needed.

## 1 · Configure the engine

In [ ]:
import os

import screamingface as sf

ENGINE_URL = os.environ.get("SCREAMINGFACE_ENGINE_URL", "http://127.0.0.1:4404")
sf.config(engine=ENGINE_URL)

`sf.config(...)` selects the one engine used by model discovery and later model
execution. It validates and stores the HTTP(S) origin but does not make a request itself.

## 2 · List executable models

In [ ]:
model_ids = sf.models.list()
model_ids

`sf.models.list()` asks the configured engine for its current capability registry
and returns executable model IDs in registry order. These are routes the engine knows how to run.

Discovery does not call a model and does not prove that provider credentials are connected. A
listed route can still fail at execution time if its provider is unavailable or unauthenticated.

## 3 · Filter the model IDs

In [ ]:
gemini_models = sf.models.list(query="gemini")
web_search_models = sf.models.list(tools=("web_search",))
first_two_models = sf.models.list(limit=2)

{
    "query=gemini": gemini_models,
    "tools=web_search": web_search_models,
    "limit=2": first_two_models,
}

The filters are small and predictable:

- `query` is a case-insensitive substring match on the model ID;
- `tools` keeps routes that advertise every requested tool; and
- `limit` returns a stable prefix after filtering.

Here, `web_search` means the engine route advertises that named executable capability. It does not
mean the model learned web content during training, and it does not validate provider access.

## 4 · List executable benchmarks

In [ ]:
benchmark_ids = sf.benchmarks.list()
benchmark_ids

`sf.benchmarks.list()` validates the same engine registry and returns its benchmark
IDs. It does not download cases. Like model discovery, it deliberately returns plain IDs rather
than introducing a second summary object.

## 5 · Filter the benchmark IDs

In [ ]:
gpqa_matches = sf.benchmarks.list(query="gpqa")
web_research_benchmarks = sf.benchmarks.list(tools=("web_search",))
first_benchmark = sf.benchmarks.list(limit=1)

{
    "query=gpqa": gpqa_matches,
    "tools=web_search": web_research_benchmarks,
    "limit=1": first_benchmark,
}

The parameter names match model discovery, but `tools` describes a different side
of compatibility. For models it means capabilities the route supports; for benchmarks it means
capabilities the benchmark requires from each answer-producing Fusion member.

## 6 · Load a manifest

A benchmark ID selects a manifest, not its dataset rows.
`sf.benchmarks.load("gpqa@1")` returns an immutable `sf.Benchmark` containing the engine-advertised
case, grader, aggregator, and tool routes. Cases remain on the engine until evaluation.

In [ ]:
gpqa = sf.benchmarks.load("gpqa@1")

{
    "id": gpqa.id,
    "title": gpqa.title,
    "grader": gpqa.grader,
    "aggregator": gpqa.aggregator,
}

This performs no dataset request and invents no substitute benchmark. During
`gpqa.evaluate(...)`, the engine loads the pinned Hugging Face source using its `HF_TOKEN`, applies
the stable slice inside URL4, runs the Recipe, grades it, and aggregates the report.

## Recap

- configure one engine with `sf.config(...)`;
- use `sf.models.list(...)` for model routes executable by that deployment;
- use `sf.benchmarks.list(...)` for benchmark manifests advertised by that deployment;
- both list APIs return plain IDs and accept `query`, `tools`, and `limit`;
- listing and loading contact the engine registry but do not fetch cases; and
- discovery performs no Fusion, model, grader, or aggregation work.

Continue to the quickstart to compose and evaluate a Fusion, or the architecture notebook to inspect
the registry and URL4 HTTP boundary.